# Week 4 · Tutorial 1 — Missing data: mechanisms and imputation

**Covers videos 4.1 – 4.3:** why data is missing · simple imputation (one column at a time) · model-based imputation (using the other columns).

**Datasets** (a weather station, hourly, 2023 — **one row is one hour**)
* `weather_station_hourly.csv` — what the station really delivered: `temp_c`, `wind_ms`, `pressure_hpa` with missing values; `reference_temp_c` from a nearby station that never fails.
* `weather_station_complete.csv` — the same hours with **nothing missing** (a backup logger). We use it only to *test* imputation methods: hide values we know, fill them back, and score the result.

*Why a value is missing decides what you may do about it.*

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
df = pd.read_csv("data/weather_station_hourly.csv", parse_dates=["timestamp"])
full = pd.read_csv("data/weather_station_complete.csv", parse_dates=["timestamp"])
print(df.shape)
print(df.isna().sum())
df.head()

## 1. Deleting incomplete rows

The simplest strategy is `dropna()`. Which rows does it remove — and is what is left still representative?

### ✏️ Your turn 1

Fill in the `TODO`s:
- `complete`: the rows of `df` without any missing value (`df.dropna()`);
- `lost_per_month`: the number of **lost** rows per calendar month (`df.index` minus `complete.index`, grouped by month of `timestamp`) — use `df.loc[~df.index.isin(complete.index), "timestamp"].dt.month.value_counts()`;
- `bias`: the mean of `reference_temp_c` in the complete rows minus the mean of `reference_temp_c` in *all* rows (2 decimals). The reference station has no gaps, so this shows whether deletion shifts the picture of the real temperature.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
complete = df.copy()
lost_per_month = pd.Series(dtype=int)
bias = np.nan
```
by
```python
complete = df.dropna()
lost_per_month = df.loc[~df.index.isin(complete.index), "timestamp"].dt.month.value_counts()
bias = complete["reference_temp_c"].mean() - df["reference_temp_c"].mean()
```

In [ ]:
complete = df.copy()               # TODO: drop rows with any missing value
lost_per_month = pd.Series(dtype=int)    # TODO
bias = np.nan                      # TODO

print("rows before / after deletion:", len(df), len(complete))
print("share of rows lost:", round(1 - len(complete) / len(df), 3))
if len(lost_per_month):
    print("month with most lost rows:", lost_per_month.idxmax(), "| lost:", lost_per_month.max())
print("bias of the reference temperature after deletion:", round(bias, 2))

## 2. Three mechanisms — and how to recognise them

* **MCAR** (missing completely at random): unrelated to anything, e.g. a lost radio packet.
* **MAR** (missing at random): depends on *other observed* variables, e.g. a station that is down in a storm.
* **MNAR** (missing not at random): depends on the *missing value itself*, e.g. a thermometer that fails when it gets too cold.

The observed data show clues: a **block** of consecutive gaps (station down), a **suspiciously round maximum** (sensor saturation), and
gaps that line up with a variable you can observe.

### ✏️ Your turn 2

Find the clues. Fill in the `TODO`s:

1. `gap_start`, `gap_hours`: the first timestamp and the length (in hours) of the **longest run of consecutive missing values** in `wind_ms`. (A run ends when a value is not missing. Hint: `is_na = df["wind_ms"].isna()`, `run_id = (is_na != is_na.shift()).cumsum()`.)
2. `max_wind`, `n_at_max`: the maximum of `wind_ms` and the number of hours *at exactly* that maximum.
3. `ref_when_missing` and `ref_when_observed`: mean `reference_temp_c` in the hours where `temp_c` is missing vs. observed (1 decimal).

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
gap_start = None
gap_hours = np.nan
max_wind = np.nan
n_at_max = np.nan
ref_when_missing = np.nan
ref_when_observed = np.nan
```
by
```python
runs = is_na.groupby(run_id).agg(["sum", "idxmax"])
longest = runs["sum"].idxmax()
gap_start = df.loc[runs.loc[longest, "idxmax"], "timestamp"]
gap_hours = int(runs["sum"].max())
max_wind = df["wind_ms"].max()
n_at_max = int((df["wind_ms"] == max_wind).sum())
ref_when_missing = df.loc[df["temp_c"].isna(), "reference_temp_c"].mean()
ref_when_observed = df.loc[df["temp_c"].notna(), "reference_temp_c"].mean()
```

In [ ]:
is_na = df["wind_ms"].isna()
run_id = (is_na != is_na.shift()).cumsum()

gap_start = None            # TODO
gap_hours = np.nan          # TODO
max_wind = np.nan           # TODO
n_at_max = np.nan           # TODO
ref_when_missing = np.nan   # TODO
ref_when_observed = np.nan  # TODO

print("longest gap in wind_ms starts:", gap_start, "| hours:", gap_hours)
print("max wind:", max_wind, "| hours at the maximum:", n_at_max)
print("mean reference temperature when temp_c is missing :", round(ref_when_missing, 1))
print("mean reference temperature when temp_c is observed:", round(ref_when_observed, 1))

When a value **cannot be recovered**: say so in the report, use the limit you know about (the sensor saturates at the maximum: the
true wind was *at least* that), and add an **indicator column** so that later steps can see where values were missing.

### ✏️ Your turn 3

Add indicator columns. Fill in the `TODO`: for `temp_c`, `wind_ms` and `pressure_hpa` create `<name>_was_missing` columns (`df[c].isna().astype(int)`).
The cell reports the number of hours in which **at least one** of the three variables was missing and the number in which **all three** were missing.

**What to change** — edit the cell exactly like this, then run it:

Change `pass` into `df[c + "_was_missing"] = df[c].isna().astype(int)`

In [ ]:
for c in ["temp_c", "wind_ms", "pressure_hpa"]:
    pass      # TODO: df[c + "_was_missing"] = ...

flags = [c for c in df.columns if c.endswith("_was_missing")]
print("indicator columns:", flags)
if flags:
    print("hours with at least one missing:", int((df[flags].sum(axis=1) >= 1).sum()))
    print("hours with all three missing   :", int((df[flags].sum(axis=1) == 3).sum()))

## 3. Simple imputation: one column at a time

How do we know which fill method is good? **Hide values whose true value we know, fill them back, and score the result.**
We hide 30 % of the temperature values of the complete logger at random and compute the RMSE (root mean squared error, in °C) of the filled values.

In [ ]:
rng = np.random.default_rng(0)
truth = full["temp_c"]
hidden = rng.random(len(full)) < 0.30
temp_hidden = truth.mask(hidden)
print("hidden values:", hidden.sum(), f"({hidden.mean():.1%})")

def rmse(filled):
    return float(np.sqrt(((filled[hidden] - truth[hidden]) ** 2).mean()))

### ✏️ Your turn 4

Score four fill methods on the hidden values. The starter has only the first two. Add the other two:
- `"ffill"`: forward fill, `temp_hidden.ffill()`
- `"interpolate"`: linear interpolation in time, `temp_hidden.interpolate()`

The cell prints the RMSE (2 decimals) of each method. Mean/median imputation uses the observed values only.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python

```
by
```python
    "ffill": temp_hidden.ffill(),
    "interpolate": temp_hidden.interpolate(),
```

In [ ]:
methods = {
    "mean": temp_hidden.fillna(temp_hidden.mean()),
    "median": temp_hidden.fillna(temp_hidden.median()),
    # TODO: "ffill": ..., "interpolate": ...
}

scores = pd.Series({name: rmse(filled) for name, filled in methods.items()}).round(2)
print(scores)
print("best method:", scores.idxmin())

### The same method, a gap of different length

Interpolating across **three hours** is reasonable; across **three days** it draws a straight line over several weather systems.
Here we hide one *block* of consecutive hours (starting at hour 4 000) and compare forward fill, interpolation and the overall mean.

### ✏️ Your turn 5

Change `gap_len` from 3 to **72** (three days) and run again. Report the RMSE of each method for both gap lengths.

In [ ]:
gap_len = 3         # TODO: 72
start = 4000

block = np.zeros(len(full), dtype=bool)
block[start:start + gap_len] = True
t_block = truth.mask(block)

def rmse_block(filled):
    return float(np.sqrt(((filled[block] - truth[block]) ** 2).mean()))

print("gap length:", gap_len, "hours")
print("ffill       :", round(rmse_block(t_block.ffill()), 2))
print("interpolate :", round(rmse_block(t_block.interpolate()), 2))
print("mean        :", round(rmse_block(t_block.fillna(t_block.mean())), 2))

## 4. Model-based imputation: use the columns you still have

`KNNImputer` fills a missing value with the average of the same column in the `k` most similar rows, where *similar* is measured over the other
columns. Two rules:

* **Distances depend on units**, so scale the columns first (`StandardScaler`).
* **Imputation is part of the model**: fit it on the training rows (the first 70 % of the hours) and apply it to the rest — never the other way round.

The function below hides 30 % of `temp_c` (the same hidden positions as before), fits on the training period and scores the filled values in the **test** period.

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

full["hour"] = full["timestamp"].dt.hour
n_train = int(0.7 * len(full))

def knn_rmse(columns, scale, k=5):
    X = full[["temp_c"] + columns].copy()
    X_hidden = X.copy()
    X_hidden.loc[hidden, "temp_c"] = np.nan
    train_part, test_part = X_hidden.iloc[:n_train], X_hidden.iloc[n_train:]
    if scale:
        sc = StandardScaler().fit(train_part)                      # statistics from the training rows only
        train_part, test_part = sc.transform(train_part), sc.transform(test_part)
    imputer = KNNImputer(n_neighbors=k).fit(train_part)             # fitted on the training rows only
    filled = imputer.transform(test_part)
    if scale:
        filled = sc.inverse_transform(filled)
    h_test = hidden[n_train:]
    return float(np.sqrt(((filled[h_test, 0] - X["temp_c"].iloc[n_train:][h_test]) ** 2).mean()))

print("rmse of the mean (test period):", round(float(np.sqrt(((temp_hidden.iloc[:n_train].mean() - truth.iloc[n_train:][hidden[n_train:]]) ** 2).mean())), 2))

### ✏️ Your turn 6

Three versions of the imputer. Run the cell as it is, then make the two changes one after the other:

1. Starter: all four other columns, **no scaling**.
2. Change `scale = True` — same columns, but scaled.
3. Then in addition change `columns = ["reference_temp_c"]` — only the column that is actually informative about `temp_c`.

Report the RMSE (2 decimals) of the three versions (use `k = 5`).

In [ ]:
columns = ["reference_temp_c", "pressure_hpa", "wind_ms", "hour"]    # TODO (step 3): ["reference_temp_c"]
scale = False                                                         # TODO (step 2): True

print("columns:", columns, "| scaled:", scale)
print("KNN imputation RMSE:", round(knn_rmse(columns, scale, k=5), 2))

## Summary

* Deleting incomplete rows is only safe for MCAR data; here it removes mostly **cold** hours.
* Look for blocks (station down), round maxima (saturation) and gaps that coincide with other variables.
* Score imputation methods by **hiding known values**; interpolation is good across short gaps and poor across long ones.
* Model-based imputation needs scaled, *informative* columns and must be fitted on the training data only.